In [ ]:
# Install necessary system packages for virtual display (apt-get) and Python packages for virtual display (pip)
!apt-get update -qq
!apt-get install -y --no-install-recommends \
    xvfb \
    ffmpeg \
    xorg-dev \
    libsdl2-dev \
    libosmesa6-dev \
    libglew-dev \
    mesa-utils

!pip install PyOpenGL pyvirtualdisplay

print("System and Python virtual display packages installed.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
xorg-dev is already the newest version (1:7.7+23ubuntu2).
libglew-dev is already the newest version (2.2.0-4).
mesa-utils is already the newest version (8.4.0-1ubuntu1).
libosmesa6-dev is already the newest version (23.2.1-1ubuntu3.1~22.04.3).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
libsdl2-dev is already the newest version (2.0.20+dfsg-2ubuntu1.22.04.1).
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.16).
0 upgraded, 0 newly installed, 0 to remove and 17 not upgraded.
System and Python virtual display packages installed.


In [ ]:
# Removed: Manual download and extraction of MuJoCo 2.1.0 binaries.
# The 'mujoco' Python package (installed via pip) typically manages its own binaries,
# and mixing versions can cause conflicts. For 'Humanoid-v5', the official 'mujoco'
# Python package is expected to handle the MuJoCo library.

In [ ]:
# Set environment variables and start virtual display
import os
from pyvirtualdisplay import Display

# Removed: os.environ['LD_LIBRARY_PATH'] setting as it pointed to manually downloaded MuJoCo 2.1.0 binaries.
# The 'mujoco' Python package (installed via pip) is expected to manage its own binaries.

# Set LD_PRELOAD to use a software OpenGL renderer (OSMesa) for headless environments.
# This must be set BEFORE any mujoco imports for it to take effect.
os.environ['LD_PRELOAD'] = '/usr/lib/x86_64-linux-gnu/libOSMesa.so.6'

# Explicitly tell MuJoCo to use the OSMesa backend for rendering
os.environ['MUJOCO_GL'] = 'osmesa'

# Explicitly force software rendering globally
os.environ['LIBGL_ALWAYS_SOFTWARE'] = '1'

# Set up virtual display
display = Display(visible=0, size=(1400, 900))
display.start()

# Explicitly set the DISPLAY environment variable after starting the virtual display
os.environ['DISPLAY'] = display.env()['DISPLAY']

# Verify the paths and display are set
print(f"LD_PRELOAD is set to: {os.environ.get('LD_PRELOAD')}")
print(f"MUJOCO_GL is set to: {os.environ.get('MUJOCO_GL')}")
print(f"LIBGL_ALWAYS_SOFTWARE is set to: {os.environ.get('LIBGL_ALWAYS_SOFTWARE')}")
print(f"DISPLAY is set to: {os.environ.get('DISPLAY')}")
print('Virtual display started successfully.')

LD_PRELOAD is set to: /usr/lib/x86_64-linux-gnu/libOSMesa.so.6
MUJOCO_GL is set to: osmesa
LIBGL_ALWAYS_SOFTWARE is set to: 1
DISPLAY is set to: :0
Virtual display started successfully.


## 2. Python Package Management

These steps ensure that `gymnasium`, `stable-baselines3`, and the official MuJoCo Python bindings are correctly installed and configured. Old conflicting packages (`mujoco-py`, `gym`) are uninstalled first.

In [ ]:
# Uninstall old 'mujoco-py' and 'gym' packages to avoid conflicts
!pip uninstall -y mujoco-py gym

print("Old 'mujoco-py' and 'gym' uninstalled.")

Old 'mujoco-py' and 'gym' uninstalled.


In [ ]:
# Install stable-baselines3 with extra dependencies (which includes gymnasium)
!pip install stable-baselines3[extra]

print("Stable-baselines3 with extras installed.")

Stable-baselines3 with extras installed.


In [ ]:
# Ensure gymnasium and the official mujoco python package are installed
!pip install gymnasium mujoco

print("Gymnasium and official mujoco python package installed.")

Gymnasium and official mujoco python package installed.


In [ ]:
# Install imageio with ffmpeg support for video recording
!pip install imageio[ffmpeg]

print("imageio[ffmpeg] installed.")

imageio[ffmpeg] installed.


## 3. Model Training, Saving, and Loading

These steps handle the core reinforcement learning workflow, including environment creation, agent training, and model persistence.

In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO

# Create the environment using gymnasium and Humanoid-v5
env = gym.make("Humanoid-v5", render_mode='rgb_array')

# Create PPO agent
model = PPO("MlpPolicy", env, verbose=1)

# Train the agent
model.learn(total_timesteps=1e6)

print("PPO model trained.")

Streaming output truncated to the last 5000 lines.
|    n_updates            | 2610        |
|    policy_gradient_loss | -0.0506     |
|    std                  | 0.861       |
|    value_loss           | 222         |
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 79.3        |
|    ep_rew_mean          | 383         |
| time/                   |             |
|    fps                  | 410         |
|    iterations           | 263         |
|    time_elapsed         | 1312        |
|    total_timesteps      | 538624      |
| train/                  |             |
|    approx_kl            | 0.039837874 |
|    clip_fraction        | 0.259       |
|    clip_range           | 0.2         |
|    entropy_loss         | -21.5       |
|    explained_variance   | 0.923       |
|    learning_rate        | 0.0003      |
|    loss                 | 69.5        |
|    n_updates           

In [ ]:
# Save the trained model
model.save("humanoid_ppo_model")

print("Model saved as 'humanoid_ppo_model'.")

Model saved as 'humanoid_ppo_model'.


In [ ]:
# Load the trained model
model = PPO.load("humanoid_ppo_model")

print("Model loaded.")

Model loaded.


In [ ]:
pip install mujoco-py gym

  Using cached mujoco_py-2.1.2.14-py3-none-any.whl.metadata (669 bytes)
  Using cached gym-0.26.2-py3-none-any.whl
Using cached mujoco_py-2.1.2.14-py3-none-any.whl (2.4 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [ ]:
# 1. Download MuJoCo 2.1.0 binaries
# As of MuJoCo 2.1.0, it is free to use.
# The file is hosted on GitHub for convenience in Colab.
!wget -q https://github.com/google-deepmind/mujoco/releases/download/2.1.0/mujoco210-linux-x86_64.tar.gz

# 2. Create the directory for MuJoCo and extract
!mkdir -p ~/.mujoco
!tar -xvf mujoco210-linux-x86_64.tar.gz -C ~/.mujoco

# The tar command already extracts to ~/.mujoco/mujoco210, so no 'mv' is needed.
# The previous 'mv' command caused an error because it tried to move a directory into itself.

mujoco210/
mujoco210/THIRD_PARTY_NOTICES
mujoco210/bin/
mujoco210/bin/libglfw.so.3
mujoco210/bin/derivative
mujoco210/bin/libglew.so
mujoco210/bin/record
mujoco210/bin/simulate
mujoco210/bin/libglewegl.so
mujoco210/bin/testxml
mujoco210/bin/basic
mujoco210/bin/libglewosmesa.so
mujoco210/bin/libglfw3.a
mujoco210/bin/libmujoco210.so
mujoco210/bin/compile
mujoco210/bin/libmujoco210nogl.so
mujoco210/bin/testspeed
mujoco210/include/
mujoco210/include/glfw3.h
mujoco210/include/uitools.c
mujoco210/include/mjxmacro.h
mujoco210/include/mjui.h
mujoco210/include/mjrender.h
mujoco210/include/mjmodel.h
mujoco210/include/mjdata.h
mujoco210/include/mujoco.h
mujoco210/include/mjvisualize.h
mujoco210/include/uitools.h
mujoco210/model/
mujoco210/model/humanoid100.xml
mujoco210/model/arm26.xml
mujoco210/model/carpet.png
mujoco210/model/softbox.xml
mujoco210/model/grid2pin.xml
mujoco210/model/sponge.png
mujoco210/model/hammock.xml
mujoco210/model/cloth.xml
mujoco210/model/grid2.xml
mujoco210/model/grid1.x

In [ ]:
# 3. Set environment variables and start virtual display
import os
from pyvirtualdisplay import Display

os.environ['LD_LIBRARY_PATH'] = os.path.join(os.path.expanduser('~'), '.mujoco', 'mujoco210', 'bin')

# Set LD_PRELOAD to use a software OpenGL renderer (OSMesa) for headless environments.
# This must be set BEFORE any mujoco imports for it to take effect.
os.environ['LD_PRELOAD'] = '/usr/lib/x86_64-linux-gnu/libOSMesa.so.6'

# Set up virtual display
display = Display(visible=0, size=(1400, 900))
display.start()

# Verify the paths and display are set (optional)
print(f"LD_LIBRARY_PATH is set to: {os.environ.get('LD_LIBRARY_PATH')}")
print(f"LD_PRELOAD is set to: {os.environ.get('LD_PRELOAD')}")
print('Virtual display started successfully.')


LD_LIBRARY_PATH is set to: /root/.mujoco/mujoco210/bin
LD_PRELOAD is set to: /usr/lib/x86_64-linux-gnu/libOSMesa.so.6
Virtual display started successfully.


In [ ]:
pip install stable-baselines3[extra]

In [ ]:
# To fix the Cython CompileError with mujoco-py, we need to switch to gymnasium and the official mujoco package.
# First, uninstall mujoco-py and the old gym.
!pip uninstall -y mujoco-py gym

# Then, install gymnasium and the official mujoco python package.
# Stable-baselines3[extra] should already have installed gymnasium, but we ensure it's present and compatible.
!pip install gymnasium mujoco

import gymnasium as gym # Import gymnasium instead of gym
from stable_baselines3 import PPO

# Create the environment using gymnasium and the Humanoid-v5 version which uses the official mujoco bindings.
# v5 is the latest version that directly uses the official mujoco Python package.
# If Humanoid-v5 encounters issues, try Humanoid-v4.
# Added render_mode='rgb_array' to enable rendering.
env = gym.make("Humanoid-v5", render_mode='rgb_array')

# Create PPO agent
model = PPO("MlpPolicy", env, verbose=1)

# Train the agent
model.learn(total_timesteps=1e6)

Streaming output truncated to the last 5000 lines.
|    loss                 | 91.6        |
|    n_updates            | 2610        |
|    policy_gradient_loss | -0.0515     |
|    std                  | 0.862       |
|    value_loss           | 279         |
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 81.9        |
|    ep_rew_mean          | 402         |
| time/                   |             |
|    fps                  | 405         |
|    iterations           | 263         |
|    time_elapsed         | 1327        |
|    total_timesteps      | 538624      |
| train/                  |             |
|    approx_kl            | 0.035001203 |
|    clip_fraction        | 0.245       |
|    clip_range           | 0.2         |
|    entropy_loss         | -21.5       |
|    explained_variance   | 0.926       |
|    learning_rate        | 0.0003      |
|    loss                

In [ ]:
from stable_baselines3 import PPO

# Load the trained model to ensure 'model' is defined
model = PPO.load("humanoid_ppo_model")

# Save the trained model
model.save("humanoid_ppo_model")